In [ ]:
import pandas as pd
import yaml
import pycountry

### File paths

In [ ]:
production_fn = '../../resources/ironore-production.csv'
bus_locations_fn = '../../data/bus_locations.csv'

# Outputs
production_clustered_fn = '../../resources/ironore_production_clustered.csv'

### Get iron ore data

In [ ]:

# Read the CSV data
production = pd.read_csv(production_fn, index_col='ISO_A2')

In [ ]:
production

### Cluster per region

### Cluster countries per region
Map each country in the production data to its region using the region definitions from the config file.

In [ ]:
# Load region definitions from config file
with open('../../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
regions = config['regions']

# Build a mapping from country name to ISO_A2 code
country_name_to_iso = {}
for country in pycountry.countries:
    country_name_to_iso[country.name] = country.alpha_2
    if hasattr(country, 'official_name'):
        country_name_to_iso[country.official_name] = country.alpha_2

# Build a mapping from ISO_A2 code to region
iso_to_region = {}
for region, countries in regions.items():
    for country in countries:
        for part in country.split('+'):
            name = part.strip()
            code = country_name_to_iso.get(name)
            if code:
                iso_to_region[code] = region

# Map each row in production to its region
production['region'] = production.index.map(lambda iso: iso_to_region.get(iso, 'Other'))

# Group by region and sum iron ore production
regional_production = production.groupby('region')['IronOreProductionMt'].sum()

# Ensure all regions are present, with 0 for missing ones
all_regions = pd.Series(0, index=regions.keys())
regional_production = all_regions.add(regional_production, fill_value=0).rename('IronOreProductionMt')

# Convert to DataFrame and sort
regional_production = regional_production.reset_index().sort_values(by='IronOreProductionMt', ascending=False)

# Rename index to region
regional_production = regional_production.rename(columns={'index': 'region'})

regional_production

### Aggregate iron ore production by region

In [ ]:
# Group by region and sum all numeric columns
production_by_region = regional_production.groupby('region').sum(numeric_only=True)
production_by_region

### Save clustered iron ore production data

In [ ]:
production_by_region.to_csv(production_clustered_fn)